# Projet 8 OCR — Analyse sociodémographique des étudiants Data
## Google Colab — Visualisations depuis le mart dbt

**Instructions :**
1. Monter Google Drive (cellule 0)
2. Le CSV est lu depuis `MyDrive/ColabProjet8/mart_profil_sociodemographique.csv`
3. Exécuter les cellules dans l'ordre
4. Télécharger les PNG : panneau Fichiers → clic droit → Télécharger

## 0. Mount Drive + Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})

BLUE  = '#1f6aa5'
CORAL = '#e05a4e'
TEAL  = '#2a9d8f'
GOLD  = '#e9c46a'
GRAY  = '#adb5bd'

print('OK')

## 1. Chargement des données

In [ ]:
CSV_PATH = '/content/drive/MyDrive/ColabProjet8/mart_profil_sociodemographique.csv'

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.lower()

print(f'Lignes : {len(df)}')
print(f'Colonnes : {list(df.columns)}')
print(f'Annees : {sorted(df.year.dropna().unique())}')
print(f'Genres : {sorted(df.gender.dropna().unique())}')
df.head()

## 2. Vérification totaux

In [ ]:
yearly_total = (
    df[df.gender == 'Total']
    .groupby('year')['student_count']
    .sum().dropna()
)
print('Etudiants par annee :')
print(yearly_total)
print(f'Total cumule : {yearly_total.sum():.0f}')

## Graphique 1 — Effectifs par région

In [ ]:
region_totals = (
    df[df.gender == 'Total']
    .groupby('region')['student_count']
    .sum().dropna()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(13, 5))
colors = [CORAL if 'France' in str(r) else BLUE for r in region_totals.index]
bars = ax.bar(region_totals.index, region_totals.values, color=colors, width=0.7)

for bar, val in zip(bars, region_totals.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{int(val):,}', ha='center', va='bottom', fontsize=8.5)

idf_val = region_totals[region_totals.index.str.contains('France', na=False)]
idf_pct = idf_val.sum() / region_totals.sum() * 100

ax.set_title(f'Effectifs OCR par region — Parcours Data 2022-2025\nIle-de-France : {idf_pct:.1f}% des inscriptions', pad=12)
ax.set_ylabel("Nombre d'etudiants")
ax.tick_params(axis='x', rotation=40)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('/content/graph_regions.png', bbox_inches='tight')
plt.show()
print('Sauvegarde : graph_regions.png')

## Graphique 2 — Évolution annuelle des inscriptions

In [ ]:
yearly = (
    df[df.gender == 'Total']
    .groupby('year')['student_count']
    .sum().dropna().sort_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(yearly.index.astype(int), yearly.values,
        marker='o', linewidth=2.5, markersize=8, color=BLUE)

for year, val in yearly.items():
    ax.annotate(f'{int(val):,}', xy=(year, val), xytext=(0, 12),
                textcoords='offset points', ha='center',
                fontsize=10, fontweight='bold', color=BLUE)

ax.set_title("Evolution des inscriptions au parcours Data OCR", pad=12)
ax.set_ylabel("Nombre d'etudiants")
ax.set_xlabel('Annee')
ax.set_xticks(yearly.index.astype(int))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, yearly.max() * 1.25)

plt.tight_layout()
plt.savefig('/content/graph_yearly.png', bbox_inches='tight')
plt.show()
print('Sauvegarde : graph_yearly.png')

## Graphique 3 — Taux de genre Non renseigné par année

In [ ]:
nr = df[df.gender == 'Non renseigne'].groupby('year')['student_count'].sum().dropna()
total_yr = df[df.gender == 'Total'].groupby('year')['student_count'].sum().dropna()
nr_rate = (nr / total_yr * 100).sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(nr_rate.index.astype(int), nr_rate.values,
              color=[CORAL, CORAL, TEAL, TEAL], width=0.6)

for bar, val in zip(bars, nr_rate.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Taux de genre non renseigne par annee\n(indicateur qualite de collecte)', pad=12)
ax.set_ylabel('% des inscriptions')
ax.set_xlabel('Annee')
ax.set_xticks(nr_rate.index.astype(int))
ax.set_ylim(0, nr_rate.max() * 1.25)
ax.axhline(y=10, color=GRAY, linestyle='--', linewidth=1, label='Seuil 10%')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/content/graph_nr_rate.png', bbox_inches='tight')
plt.show()
print('Sauvegarde : graph_nr_rate.png')

## Graphique 4 — Benchmark INSEE : étudiants pour 100 000 habitants

In [ ]:
bench = (
    df[
        (df.gender == 'Total') &
        (~df.region.isin(['DROM', 'Corse'])) &
        (df.students_per_100k.notna())
    ]
    .groupby('region')['students_per_100k']
    .mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(13, 5))
colors = [CORAL if 'France' in str(r) else TEAL for r in bench.index]
bars = ax.bar(bench.index, bench.values, color=colors, width=0.7)

for bar, val in zip(bars, bench.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8.5)

mean_val = bench.mean()
ax.axhline(y=mean_val, color=GOLD, linestyle='--', linewidth=1.5,
           label=f'Moyenne nationale : {mean_val:.1f}')
ax.set_title("Taux d'inscription pour 100 000 habitants par region\n(moyenne 2022-2025, hors DROM et Corse)", pad=12)
ax.set_ylabel('Etudiants OCR / 100 000 hab.')
ax.tick_params(axis='x', rotation=40)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/content/graph_100k.png', bbox_inches='tight')
plt.show()
print('Sauvegarde : graph_100k.png')

## Graphique bonus — Répartition H/F par année

In [ ]:
gender_known = (
    df[df.gender.isin(['M', 'F'])]
    .groupby(['year', 'gender'])['student_count']
    .sum().dropna()
    .unstack(fill_value=0)
    .sort_index()
)
gender_pct = gender_known.div(gender_known.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 4))
x = gender_pct.index.astype(int)
ax.bar(x, gender_pct['M'], label='Hommes', color=BLUE, width=0.5)
ax.bar(x, gender_pct['F'], bottom=gender_pct['M'], label='Femmes', color=CORAL, width=0.5)

for year in x:
    m = gender_pct.loc[year, 'M']
    f = gender_pct.loc[year, 'F']
    ax.text(year, m/2, f'{m:.0f}%', ha='center', va='center',
            color='white', fontsize=10, fontweight='bold')
    ax.text(year, m + f/2, f'{f:.0f}%', ha='center', va='center',
            color='white', fontsize=10, fontweight='bold')

ax.set_title('Repartition H/F par annee\n(genre renseigne uniquement)', pad=12)
ax.set_ylabel('%')
ax.set_xticks(x)
ax.set_ylim(0, 105)
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('/content/graph_gender_split.png', bbox_inches='tight')
plt.show()
print('Sauvegarde : graph_gender_split.png')

## Résumé des fichiers produits

In [ ]:
import os
pngs = sorted([f for f in os.listdir('/content') if f.endswith('.png')])
print('Fichiers PNG produits :')
for f in pngs:
    print(f'  v {f}')
print('\nTelecharger : panneau Fichiers (gauche) → clic droit → Telecharger')